# 5. Pseudoinverse  [4.5]

TOML-MIRI — Linear Algebra for Data Science, exploration exercises.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from ipywidgets import interact, FloatSlider

np.set_printoptions(precision=4, suppress=True)

def check(label, ok):
    print(f"{'✓' if ok else '✗'}  {label}")

---
## 5.1 Three roads to $A^\dagger$

**Task.** Compute $A^\dagger$ for a tall full-column-rank matrix, a wide full-row-rank matrix, and a rank-deficient square matrix, in the three ways below, and compare with `np.linalg.pinv`.

$$A^\dagger_{\text{left}} = (A^\top A)^{-1}A^\top, \qquad A^\dagger_{\text{right}} = A^\top(AA^\top)^{-1}, \qquad A^\dagger = V\Sigma^\dagger U^\top,$$
$$\Sigma^\dagger = \mathrm{diag}(1/\sigma_1, \dots, 1/\sigma_r, 0, \dots, 0) \in \mathbb{R}^{n\times m}.$$

The matrices:
- **tall** — Example 1.14 / 4.2: $\begin{bmatrix}2&1\\1&3\\0&1\end{bmatrix}$
- **wide** — Example 1.13: $\begin{bmatrix}2&1&0\\1&3&1\end{bmatrix}$
- **square rank-deficient**: $\begin{bmatrix}1&2&3\\4&5&6\\7&8&9\end{bmatrix}$ (rank 2)

### Prediction (write this *before* running)

- For each matrix, which of the three formulas will work?
- When a formula 'fails', will NumPy raise an error or silently return garbage?

*(fill in)*

In [ ]:
A_tall = np.array([[2., 1.], [1., 3.], [0., 1.]])
A_wide = np.array([[2., 1., 0.], [1., 3., 1.]])
A_sing = np.array([[1., 2., 3.], [4., 5., 6.], [7., 8., 9.]])
mats = {"tall 3×2": A_tall, "wide 2×3": A_wide, "square rank-2 3×3": A_sing}

def pinv_left(A):
    return np.linalg.inv(A.T @ A) @ A.T

def pinv_right(A):
    return A.T @ np.linalg.inv(A @ A.T)

def pinv_svd(A):
    U, s, Vt = np.linalg.svd(A)
    m, n = A.shape
    r = int(np.sum(s > s.max() * max(m, n) * np.finfo(float).eps))
    Sigma_dag = np.zeros((n, m))
    Sigma_dag[:r, :r] = np.diag(1 / s[:r])
    return Vt.T @ Sigma_dag @ U.T

methods = {"left  (AᵀA)⁻¹Aᵀ": (pinv_left, lambda A: A.T @ A),
           "right Aᵀ(AAᵀ)⁻¹": (pinv_right, lambda A: A @ A.T),
           "SVD   VΣ†Uᵀ": (pinv_svd, None)}

rows = []
for mname, A in mats.items():
    ref = np.linalg.pinv(A)
    for meth, (f, inner) in methods.items():
        cond = f"{np.linalg.cond(inner(A)):.2e}" if inner else "—"
        try:
            Ad = f(A)
            ok = np.allclose(Ad, ref, atol=1e-8)
            result = "matches pinv" if ok else f"WRONG  (largest entry {np.abs(Ad).max():.2e})"
        except np.linalg.LinAlgError as err:
            result = f"error: {err}"
        rows.append({"matrix": mname, "rank": np.linalg.matrix_rank(A), "method": meth,
                     "cond of the inverted matrix": cond, "result": result})
pd.DataFrame(rows)

### Verify against Example 4.2

In [ ]:
Ad_notes = np.array([[0.567, -0.133, -0.167], [-0.167, 0.333, 0.167]])
check("A† (tall) matches Example 4.2", np.allclose(pinv_svd(A_tall), Ad_notes, atol=1e-3))
check("A† b = [0.9, 0.5] for b = [3, 1, 4]", np.allclose(pinv_svd(A_tall) @ [3, 1, 4], [0.9, 0.5]))
for name, A in mats.items():
    check(f"{name}: SVD formula = np.linalg.pinv", np.allclose(pinv_svd(A), np.linalg.pinv(A)))

---
### Explain (hand-in — in your own words)

**1. Which formulas fail for which matrix, and why?**
*(Hint: what are the size and rank of $A^\top A$ and of $AA^\top$ in each case? A square matrix is invertible only if it has full rank.)*

*Your answer.*

**2. Which one always works? Why does it not need any inverse of a possibly singular matrix?**

*Your answer.*

---
## 5.2 Left vs right inverse

**Task.** For the matrices in the previous exercise, compute $A^\dagger A$ and $AA^\dagger$.

### Prediction (write this *before* running)

- For the tall matrix, which of $A^\dagger A$ and $AA^\dagger$ is the identity? For the wide one?
- For the rank-deficient square matrix?

*(fill in)*

In [ ]:
def describe(P, A):
    # compare P with I, and with the orthogonal projectors onto RS(A) (n×n) and CS(A) (m×m)
    U, s, Vt = np.linalg.svd(A)
    r = np.linalg.matrix_rank(A)
    k = P.shape[0]
    P_RS = Vt[:r].T @ Vt[:r]
    P_CS = U[:, :r] @ U[:, :r].T
    return {"size": f"{k}×{k}",
            "= I": np.allclose(P, np.eye(k)),
            "P² = P": np.allclose(P @ P, P),
            "P = Pᵀ": np.allclose(P, P.T),
            "rank": np.linalg.matrix_rank(P),
            "= proj onto RS(A)": k == A.shape[1] and np.allclose(P, P_RS),
            "= proj onto CS(A)": k == A.shape[0] and np.allclose(P, P_CS)}

rows = []
for name, A in mats.items():
    Ad = np.linalg.pinv(A)
    rows.append({"matrix": name, "product": "A†A", **describe(Ad @ A, A)})
    rows.append({"matrix": name, "product": "AA†", **describe(A @ Ad, A)})
pd.DataFrame(rows)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))
for j, (name, A) in enumerate(mats.items()):
    Ad = np.linalg.pinv(A)
    for i, (label, P) in enumerate([("A†A", Ad @ A), ("AA†", A @ Ad)]):
        ax = axes[i, j]
        ax.imshow(P, cmap="RdBu_r", vmin=-1, vmax=1)
        for (a, b), val in np.ndenumerate(P):
            ax.text(b, a, f"{val:.2f}", ha="center", va="center")
        ax.set_title(f"{name}: {label}"); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

### Verify numerically

In [ ]:
P_notes = np.array([[29, 2, -5], [2, 26, 10], [-5, 10, 5]]) / 30           # Example 2.2: same A as A_tall
check("tall: AA† = A(AᵀA)⁻¹Aᵀ = P of Example 2.2", np.allclose(A_tall @ np.linalg.pinv(A_tall), P_notes))
for name, A in mats.items():
    Ad = np.linalg.pinv(A)
    check(f"{name}: A A† A = A   and   A† A A† = A†", np.allclose(A @ Ad @ A, A) and np.allclose(Ad @ A @ Ad, Ad))

---
### Explain (hand-in — in your own words)

**1. Which product is the identity in each case?**

*Your answer.*

**2. The other one is a projection matrix: check $P^2 = P = P^\top$ and say onto which subspace (row space or column space) it projects [2.3, 4.5].**
*(Hint: $A^\dagger A$ acts on $\mathbb{R}^n$ (inputs), $AA^\dagger$ on $\mathbb{R}^m$ (outputs). Which of the four subspaces live there?)*

*Your answer.*

---
## 5.3 Projection in pictures

**Task.** Take $A$ from Example 2.2 ($3\times2$). Plot its column space as a plane in 3-D, a vector $t$, its projection $\hat t$ and the residual $e$:

$$\hat t = Pt = A(A^\top A)^{-1}A^\top t, \qquad e = t - \hat t.$$

*Note:* the notes write $t = [1, 2]$, but $P$ is $3\times3$, so $t$ must live in $\mathbb{R}^3$. Use the sliders.

### Prediction (write this *before* running)

- What angle does $e$ make with the plane?
- What happens to $\hat t$ and $e$ when $t$ already lies in the plane? When $t$ is perpendicular to it?

*(fill in)*

In [ ]:
A = A_tall
P = A @ np.linalg.inv(A.T @ A) @ A.T
U, s, Vt = np.linalg.svd(A)
cs_basis, lns = U[:, :2], U[:, 2]

def arrow(start, end, color, name):
    d = end - start
    return [go.Scatter3d(x=[start[0], end[0]], y=[start[1], end[1]], z=[start[2], end[2]], mode="lines",
                         line=dict(color=color, width=7), name=name),
            go.Cone(x=[end[0]], y=[end[1]], z=[end[2]], u=[d[0]], v=[d[1]], w=[d[2]], sizemode="absolute",
                    sizeref=0.25, anchor="tip", colorscale=[[0, color], [1, color]], showscale=False,
                    showlegend=False, hoverinfo="skip")]

def plot_projection(t1, t2, t3):
    t = np.array([t1, t2, t3])
    t_hat = P @ t
    e = t - t_hat
    L = 4
    g = np.linspace(-L, L, 2)
    S1, S2 = np.meshgrid(g, g)
    plane = cs_basis[:, [0]] * S1.ravel() + cs_basis[:, [1]] * S2.ravel()
    o = np.zeros(3)
    fig = go.Figure([go.Surface(x=plane[0].reshape(2, 2), y=plane[1].reshape(2, 2), z=plane[2].reshape(2, 2),
                                colorscale=[[0, "seagreen"], [1, "seagreen"]], opacity=0.3, showscale=False,
                                name="CS(A)", showlegend=True)]
                    + arrow(o, A[:, 0], "seagreen", "a₁") + arrow(o, A[:, 1], "darkgreen", "a₂")
                    + arrow(o, t, "black", "t") + arrow(o, t_hat, "royalblue", "t̂ = Pt")
                    + arrow(t_hat, t, "crimson", "e = t − t̂"))
    rng_ = dict(range=[-L, L])
    fig.update_layout(scene=dict(xaxis=rng_, yaxis=rng_, zaxis=rng_, aspectmode="cube"), height=600)
    fig.show()
    print(f"t̂ = {t_hat}   e = {e}   Aᵀe = {A.T @ e}   ‖e‖ = {np.linalg.norm(e):.4f}")

@interact(t1=FloatSlider(min=-3, max=3, step=0.1, value=1.0),
          t2=FloatSlider(min=-3, max=3, step=0.1, value=2.0),
          t3=FloatSlider(min=-3, max=3, step=0.1, value=3.0))
def explore_projection(t1, t2, t3):
    plot_projection(t1, t2, t3)

### Verify numerically

In [ ]:
t = np.random.default_rng(0).standard_normal(3)
e = t - P @ t
check("P = 1/30 [[29, 2, −5], [2, 26, 10], [−5, 10, 5]]  (Example 2.2)", np.allclose(P, P_notes))
check("P² = P = Pᵀ", np.allclose(P @ P, P) and np.allclose(P, P.T))
check("rank P = 2 (= rank A)", np.linalg.matrix_rank(P) == 2)
check("Aᵀ e = 0  (e ⟂ both columns of A)", np.allclose(A.T @ e, 0))
check("e is parallel to u₃, the basis of LNS(A)", np.isclose(abs(e @ lns), np.linalg.norm(e)))
check("t̂ = A x̂ with x̂ the least-squares solution", np.allclose(P @ t, A @ np.linalg.lstsq(A, t, rcond=None)[0]))
check("‖t‖² = ‖t̂‖² + ‖e‖²  (Pythagoras)", np.isclose(t @ t, (P @ t) @ (P @ t) + e @ e))

---
### Explain (hand-in — in your own words)

**1. Why is the residual always perpendicular to the plane?**
*(Hint: write out $A^\top e = A^\top t - A^\top A (A^\top A)^{-1} A^\top t$.)*

*Your answer.*

**2. Which of the four subspaces does it live in?**

*Your answer.*